In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/soil-grain-size-from-photos/sample_submission.csv
/kaggle/input/competitions/soil-grain-size-from-photos/Training_labels_updated.csv
/kaggle/input/competitions/soil-grain-size-from-photos/ppm_updated.csv
/kaggle/input/competitions/soil-grain-size-from-photos/Training-All_Photos_updated/Training-All_Photos_updated/Samsung_A52_H666_01.jpg
/kaggle/input/competitions/soil-grain-size-from-photos/Training-All_Photos_updated/Training-All_Photos_updated/Motorola_Edge_60_fusion_H374_02.jpg
/kaggle/input/competitions/soil-grain-size-from-photos/Training-All_Photos_updated/Training-All_Photos_updated/Motorola_Edge_60_fusion_H374_03.jpg
/kaggle/input/competitions/soil-grain-size-from-photos/Training-All_Photos_updated/Training-All_Photos_updated/Motorola_Edge_H368_04.jpg
/kaggle/input/competitions/soil-grain-size-from-photos/Training-All_Photos_updated/Training-All_Photos_updated/Samsung_A52_H038_01.jpg
/kaggle/input/competitions/soil-grain-size-from-photos/Training-All_

In [2]:
import os
import re
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

# --- NEW IMPORTS FOR CNN ---
import torch
import torchvision.models as models
import torchvision.transforms as transforms

DATA_DIR = "/kaggle/input/competitions/soil-grain-size-from-photos"
OUTPUT_DIR = "/kaggle/working"

TRAIN_IMG_DIR = os.path.join(DATA_DIR, "Training-All_Photos_updated", "Training-All_Photos_updated")
TEST_IMG_DIR = os.path.join(DATA_DIR, "Test_All_Photos", "Test_All_Photos")

## Step 1 — Load Data
labels = pd.read_csv(os.path.join(DATA_DIR, "Training_labels_updated.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

## Step 2 — Set Targets
id_col_candidates = [c for c in labels.columns if re.search(r"id|sample", c, re.I)]
sample_id_col = id_col_candidates[0]
target_cols = [c for c in labels.columns if c != sample_id_col]

## Step 3 — Setup Pre-trained ResNet Feature Extractor
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load a pre-trained ResNet18
weights = models.ResNet18_Weights.IMAGENET1K_V1
resnet = models.resnet18(weights=weights)

# Replace the final classification layer with an Identity layer 
# so it outputs the raw 512-dimensional feature vector instead of class probabilities.
resnet.fc = torch.nn.Identity()
resnet = resnet.to(device)
resnet.eval()

# Standard ImageNet preprocessing pipeline
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def extract_cnn_features(path):
    """Passes an image through ResNet18 and returns a 512D numpy array."""
    img = Image.open(path).convert("RGB")
    input_tensor = preprocess(img)
    input_batch = input_tensor.unsqueeze(0).to(device) # Create a mini-batch of size 1

    with torch.no_grad():
        features = resnet(input_batch)
    
    return features.cpu().numpy().flatten()

# ResNet18 outputs 512 features
FEATURE_NAMES = [f"cnn_feat_{i}" for i in range(512)]

## Step 4 — Extract CNN Features for Training Photos
train_photo_files = sorted(os.listdir(TRAIN_IMG_DIR))

def filename_to_sample_id(filename):
    match = re.search(r"([A-Z]\d{3})", filename)
    return match.group(1) if match else None

rows = []
for i, fname in enumerate(train_photo_files):
    sid = filename_to_sample_id(fname)
    if sid is None:
        continue
    
    feats = extract_cnn_features(os.path.join(TRAIN_IMG_DIR, fname))
    rows.append([sid] + list(feats))
    
    if (i + 1) % 100 == 0:
        print(f"Extracted features for {i + 1} / {len(train_photo_files)} train images...")

photo_features = pd.DataFrame(rows, columns=["sample_id"] + FEATURE_NAMES)
# Average the 512D vectors for photos belonging to the same sample
sample_features = photo_features.groupby("sample_id")[FEATURE_NAMES].mean().reset_index()

## Step 5 — Merge features with labels
merged = sample_features.merge(
    labels, left_on="sample_id", right_on=sample_id_col, how="inner"
)

X = merged[FEATURE_NAMES].values
Y = merged[target_cols].values
groups = merged["sample_id"].values

## Step 6 — Train + Validate Random Forest on CNN Embeddings
def emd_1d(curve_a, curve_b):
    return np.sum(np.abs(np.array(curve_a) - np.array(curve_b)))

gkf = GroupKFold(n_splits=8)
fold_scores = []

# Random Forest parameters adjusted for higher dimensionality
model = RandomForestRegressor(n_estimators=300, max_depth=15, random_state=42, n_jobs=-1)

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, Y, groups)):
    X_train_fold, X_val_fold = X[train_idx], X[val_idx]
    Y_train_fold, Y_val_fold = Y[train_idx], Y[val_idx]
    
    model.fit(X_train_fold, Y_train_fold)
    preds = model.predict(X_val_fold)

    fold_emds = [emd_1d(preds[i], Y_val_fold[i]) for i in range(len(preds))]
    
    fold_score = np.mean(fold_emds)
    fold_scores.append(fold_score)
    print(f"Fold {fold}: mean EMD = {fold_score:.4f}")

print(f"\nOverall CV EMD: {np.mean(fold_scores):.4f} (lower is better)")

## Step 7 — Extract CNN Features for TEST Photos
def filename_to_sample_id_test(filename):
    name = os.path.splitext(filename)[0]
    name = re.sub(r"\s*\(\d+\)\s*$", "", name)            
    match = re.search(r"(HPC_.*)$", name)                 
    if match:
        name = match.group(1)
    name = name.replace("ü", "ue").replace(",", "_")      
    return name

test_photo_files = sorted(os.listdir(TEST_IMG_DIR))
test_rows = []
for i, fname in enumerate(test_photo_files):
    sid = filename_to_sample_id_test(fname)
    if sid is None:
        continue
    
    feats = extract_cnn_features(os.path.join(TEST_IMG_DIR, fname))
    test_rows.append([sid] + list(feats))
    
    if (i + 1) % 100 == 0:
        print(f"Extracted features for {i + 1} / {len(test_photo_files)} test images...")

test_photo_features = pd.DataFrame(test_rows, columns=["sample_id"] + FEATURE_NAMES)
test_sample_features = test_photo_features.groupby("sample_id")[FEATURE_NAMES].mean().reset_index()

# Final prediction
X_test = test_sample_features[FEATURE_NAMES].values
model.fit(X, Y)  # Retrain on all available data
test_predictions = model.predict(X_test)

## Step 8 — Build Submission
final_sub = sample_sub.copy()
pred_df = pd.DataFrame(test_predictions, columns=target_cols)
pred_df["sample_id"] = test_sample_features["sample_id"].values

final_sub = final_sub.merge(
    pred_df, left_on=sample_id_col, right_on="sample_id",
    how="left", suffixes=("", "_pred")
)
for col in target_cols:
    if f"{col}_pred" in final_sub.columns:
        final_sub[col] = final_sub[f"{col}_pred"]
final_sub = final_sub[sample_sub.columns]  

final_sub.to_csv(os.path.join(OUTPUT_DIR, "submission_resnet_rf.csv"), index=False)
print("Saved submission_resnet_rf.csv — ready for submission!")

Using device: cpu
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 158MB/s]


Extracted features for 100 / 127 train images...
Fold 0: mean EMD = 101.8810
Fold 1: mean EMD = 98.9318
Fold 2: mean EMD = 88.6954
Fold 3: mean EMD = 113.2358
Fold 4: mean EMD = 124.4295
Fold 5: mean EMD = 53.5103
Fold 6: mean EMD = 91.3469
Fold 7: mean EMD = 127.1293

Overall CV EMD: 99.8950 (lower is better)
Saved submission_resnet_rf.csv — ready for submission!
